In [15]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [16]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

4-element Vector{Int64}:
 10
 11
 12
 13

In [17]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 300
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "ARp"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [18]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [19]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

      From worker 13:	Precompiling TvPersistence...
      From worker 10:	Precompiling TvPersistence...
      From worker 11:	Precompiling TvPersistence...
      From worker 12:	Precompiling TvPersistence...
      From worker 13:	    TvPersistence Being precompiled by another process (pid: 24580, pidfile: C:\Users\mikul\.julia\compiled\v1.11\TvPersistence\3xGnn_3Op3r.ji.pidfile)
      From worker 11:	    TvPersistence Being precompiled by another process (pid: 24580, pidfile: C:\Users\mikul\.julia\compiled\v1.11\TvPersistence\3xGnn_3Op3r.ji.pidfile)
      From worker 12:	    TvPersistence Being precompiled by another process (pid: 24580, pidfile: C:\Users\mikul\.julia\compiled\v1.11\TvPersistence\3xGnn_3Op3r.ji.pidfile)
      From worker 10:	   3822.8 ms  ✓ TvPersistence
      From worker 10:	  1 dependency successfully precompiled in 7 seconds. 89 already precompiled.
      From worker 12:	   4082.2 ms  ✓ TvPersistence
      From worker 12:	  1 dependency successfully precompiled in 7

In [20]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [21]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 10:	[ Info: Performing boostrap simulation number 1
      From worker 11:	[ Info: Performing boostrap simulation number 2
      From worker 13:	[ Info: Performing boostrap simulation number 4
      From worker 12:	[ Info: Performing boostrap simulation number 3
      From worker 10:	[ Info: Bootstrap 1 generated.
      From worker 11:	[ Info: Bootstrap 2 generated.
      From worker 12:	[ Info: Bootstrap 3 generated.
      From worker 13:	[ Info: Bootstrap 4 generated.


Task (done) @0x0000020f1beab270

In [22]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, 1.483388074206392e-5, -2.535895507780549e-5, 2.7945127293300193e-5, 3.741317809362053e-5, 3.155039418768317e-5, 9.993245784443729e-6, -0.00010102100628869009, -9.459283278020885e-5, -5.293128422708704e-5  …  1.4477500327089021e-5, 1.607507913433244e-5, 1.4721794144914797e-5, 1.646304294524703e-5, 1.5232554603322474e-5, 1.1414065340936312e-5, 8.269714897194799e-6, 7.200130436769342e-6, 7.6285594307383016e-6, 1.0710561555743987e-5]
 [NaN, -1.732523160520523e-5, -0.0002405075744883507, 8.643105685650419e-5, -4.2708349800998644e-5, -5.080458072703823e-5, -9.757382434524272e-5, -7.785680656279127e-5, -6.025297546224375e-5, 1.071833205669944e-5  …  -1.1786934600931895e-5, -8.407045161094553e-6, -5.941070939095719e-6, -4.936840362598297e-6, -2.482510791351647e-6, -5.340237342961392e-6, -3.061328959432153e-6, -1.214014140319997e-6, 4.432215690974156e-7, 2.3488825687079475e-6]
 [NaN, -0.000129244414845531, -9.499503550526711e-5, -0.00015177554456586133,

In [23]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [24]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 1.1040827577120086e-5


In [ ]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr